# **Student Performance Prediction Project**

In [ ]:
# ==============================
# STUDENT PERFORMANCE PREDICTION
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
#Metrics And Library

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


In [ ]:
# ==============================
# 1. LOAD DATASET
# ==============================

df = pd.read_csv("/content/Student_Performance.csv")

print("Dataset Shape:", df.shape)
print("\nFirst 5 Rows:")
print(df.head())

In [ ]:
# Remove student ID
if 'student_id' in df.columns:
    df.drop('student_id', axis=1, inplace=True)

# Handle missing values
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

# Convert all categorical columns to numeric
df = pd.get_dummies(df, drop_first=True)

# Convert boolean columns to integer
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)


In [ ]:
# ==============================
# 2. DATASET INFO
# ==============================

print("\nDataset Information:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())


In [ ]:
# ==============================
# 3. HANDLE MISSING VALUES
# ==============================

for col in df.columns:
    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

In [ ]:
# ==============================
# 4. REMOVE STUDENT ID
# ==============================

if "student_id" in df.columns:
    df.drop("student_id", axis=1, inplace=True)

In [ ]:
# ==============================
# 5. ENCODE CATEGORICAL COLUMNS
# ==============================

categorical_cols = [
    "gender",
    "school_type",
    "parent_education",
    "internet_access",
    "extra_activities",
    "study_method"
]

existing_cols = [col for col in categorical_cols if col in df.columns]

df = pd.get_dummies(
    df,
    columns=existing_cols,
    drop_first=True
)

In [ ]:

# ==============================
# 6. FEATURES & TARGET
# ==============================

X = df.drop(
    ["overall_score", "final_grade"],
    axis=1,
    errors="ignore"
)

y = df["overall_score"]

In [ ]:
# ==============================
# 7. TRAIN TEST SPLIT
# ==============================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("\nTraining Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)


In [ ]:
# ==============================
# 8. TRAIN MODEL
# ==============================

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42
)

model.fit(X_train, y_train)

In [ ]:
# ==============================
# 9. PREDICTIONS
# ==============================

y_pred = model.predict(X_test)

In [ ]:
# ==============================
# 10. EVALUATION
# ==============================

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("\n===== MODEL PERFORMANCE =====")
print("MAE :", round(mae, 2))
print("MSE :", round(mse, 2))
print("RMSE:", round(rmse, 2))
print("R² Score:", round(r2, 4))

In [ ]:
# ==============================
# 11. ACTUAL VS PREDICTED
# ==============================

results = pd.DataFrame({
    "Actual Score": y_test,
    "Predicted Score": y_pred
})

print("\nSample Predictions:")
print(results.head(10))

In [ ]:
# ==============================
# 12. FEATURE IMPORTANCE
# ==============================

importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print("\nTop Important Features:")
print(importance.head(10))

In [ ]:
# ==============================
# 13. PLOT FEATURE IMPORTANCE
# ==============================

plt.figure(figsize=(10,6))

plt.barh(
    importance["Feature"][:10],
    importance["Importance"][:10]
)

plt.xlabel("Importance")
plt.ylabel("Features")
plt.title("Top 10 Important Features")
plt.gca().invert_yaxis()

plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# 14. PREDICT NEW STUDENT
# ==============================

new_student = X.iloc[[0]].copy()

prediction = model.predict(new_student)

print("\nPredicted Overall Score:")
print(round(prediction[0], 2))

In [ ]:
# ==============================
# 15. SAVE MODEL (OPTIONAL)
# ==============================

import joblib

joblib.dump(
    model,
    "student_performance_model.pkl"
)

print("\nModel Saved Successfully!")